# Benchmarking And Profiling

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
import jax.numpy as jnp

from cthree.propagation.ScipyExpmGOAT import ScipyExpmGOAT
from cthree.propagation.ScipyExpmJax import ScipyExpmJax
from cthree.model.ClosedModel import ClosedModel
from cthree.model.Hamiltonian import Hamiltonian
from cthree.measurement.StateTransferFidelity import StateTransferFidelity, StateTransferFidelityAD
from cthree.OptimisationMap import OptimisationMap
from cthree.ScipyOptimiserGradient import ScipyOptimiserGradient
from cthree.signal.SimpleGenerator import CosGenerator, CosGeneratorAD
from cthree.signal.Device import CosToneErf, CosToneErfAD

import cProfile
import pstats

## 1. Setup

In [ ]:
t_final = 10e-9
ts = np.linspace(0.0, t_final, 100)
FREQ = 4.8e9 * 2 * np.pi

init = np.array([[1.0], [0]])    # |0>
target = np.array([[0.0], [1]])  # |1>

RES = 1000e9 # 10x larger resolution to make it more computationally demanding

def setup_experiment(useJax=False, res=RES):
    if useJax:
        tone = CosToneErfAD()
        gen = CosGeneratorAD(devices=[tone])
    else:
        tone = CosToneErf()
        gen = CosGenerator(devices=[tone])

    params = tone.getParameters()

    sigmaZ = np.array([[1.0, 0], [0, -1]])
    sigmaX = np.array([[0.0, 1], [1, 0]])

    drift = FREQ / 2 * sigmaZ

    controlled_qubit = Hamiltonian(subsystems=[drift], drives=[sigmaX], generator=gen)
    model = ClosedModel(controlled_qubit)

    params[0].setValue(0.8 * np.pi / t_final)
    params[1].setValue(1.01 * FREQ)

    if useJax:
        prop = ScipyExpmJax(model, res=res)
        zeroone = StateTransferFidelityAD(
            propagation=prop,
            initialState=init,
            targetState=target,
            times=jnp.array([0.0, t_final]),
        )
    else:
        prop = ScipyExpmGOAT(model, res=res)
        zeroone = StateTransferFidelity(
        propagation=prop,
        initialState=init,
        targetState=target,
        times=np.array([0.0, t_final]),
    )

    optmap = OptimisationMap()
    optmap.add(tone, params[0:2])
    opt = ScipyOptimiserGradient(zeroone, optimisables=optmap)
    opt.getOptimisables()

    return tone, gen, model, prop, zeroone, opt

In [ ]:
def make_plot(prop, gen):
    ts = np.linspace(0, t_final, 1001)
    states = np.reshape(prop.propagate(ts), (-1, 2))
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts/1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts/1e-9, np.abs(states) ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax

In [ ]:
tone, gen, model, prop, zeroone, opt = setup_experiment(useJax=False)
toneAD, genAD, modelAD, propAD, zerooneAD, optAD = setup_experiment(useJax=True)

## 2. Benchmarking Propagation

### 2.1. Benchmarking `model.getMatrixEOM`

For one evaluation

In [ ]:
%timeit model.getMatrixEOM(1e-9)
%timeit modelAD.getMatrixEOM(1e-9)

Evaluating in a for loop

In [ ]:
def test_getMatrixEOM(model, ts):
    for t in ts:
        model.getMatrixEOM(jnp.reshape(t, (-1,1)))

%timeit test_getMatrixEOM(model, ts)
%timeit test_getMatrixEOM(modelAD, ts)

Evaluating using vmap

In [ ]:
from jax import vmap
%timeit vmap(modelAD.getMatrixEOM)(jnp.reshape(ts, (-1,1)))

### 2.2. Benchmarking `propagate`

In [ ]:
%timeit prop.propagate(ts)
%timeit propAD.propagate(ts)

In [ ]:
%timeit prop.propagate([0.0, t_final])
%timeit propAD.propagate([0.0, t_final])

### 2.3. Benchmarking `_propagatePsi`

For jitted functions we have to use `block_until_ready()` to get the actual runtime

In [ ]:
dt = ts[1] - ts[0]
psi_t = init
eom = model.getMatrixEOM(jnp.reshape(1e-9, (-1, 1)) + dt/2) * dt
%timeit prop._propagatePsi(eom, psi_t).block_until_ready()
%timeit propAD._propagatePsi(eom, psi_t).block_until_ready()

### 2.4. Testing output of propagation

In [ ]:
make_plot(prop, gen)
make_plot(propAD, genAD)

Testing with changing the parameters

In [ ]:
FREQ = 4.8e9 * 2 * np.pi
T_FINAL = 10e-9
RES = 100e9

params = tone.getParameters()[0:2]
params[0].setValue(0.8 * np.pi / T_FINAL)
params[1].setValue(0.95 * FREQ)

paramsAD = toneAD.getParameters()[0:2]
paramsAD[0].setValue(0.8 * np.pi / T_FINAL)
paramsAD[1].setValue(0.95 * FREQ)


make_plot(prop, gen)
make_plot(propAD, genAD)

## 3. Benchmarking `prop.gradient()`

### 3.1. Benchmarking gradient propagation

In [ ]:
%timeit prop.gradient(ts)
%timeit propAD.gradient(ts)

In [ ]:
%timeit prop.gradient([0.0, t_final])
%timeit propAD.gradient([0.0, t_final])

### 3.2. Testing state propagation from gradients

In [ ]:
def make_plot_gradient(prop, gen):
    ts = np.linspace(0, t_final, 1001)
    states = np.reshape(prop.gradient(ts)[0], (-1, 2))
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts/1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts/1e-9, np.abs(states) ** 2)
    ax[1].set_ylabel("Population")
    # ax[-1].legend(np.round(np.diag(chip.getDrift()) / 2e9 / np.pi, 2))
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax

make_plot_gradient(prop, gen)
make_plot_gradient(propAD, genAD)

## 4. Benchmark Measure function

In [ ]:
%timeit zeroone.measure()
%timeit zerooneAD.measure()

In [ ]:
%timeit zeroone.measureWithGradient()
%timeit zerooneAD.measureWithGradient()

## 5. Profiling Optimization

#### 5.1. Without AD

In [ ]:
cProfile.run("opt.optimise()", "restats")

In [ ]:
p = pstats.Stats('restats')
p.sort_stats("tottime").print_stats("cthree")

#### 5.2. With AD

In [ ]:
cProfile.run("optAD.optimise()", "restats")

In [ ]:
p = pstats.Stats('restats')
p.sort_stats("tottime").print_stats("cthree")

The above doesnt capture all the jitted functions

In [ ]:
p = pstats.Stats('restats')
p.sort_stats("tottime").print_stats()

The above also has larger compile time. Lets increase the resolution further to benchmark internal function evaluation

In [ ]:
toneAD, genAD, modelAD, propAD, zerooneAD, optAD = setup_experiment(useJax=True, res=10000e9)

In [ ]:
cProfile.run("optAD.optimise()", "restats")

In [ ]:
p = pstats.Stats('restats')
p.sort_stats("tottime").print_stats("cthree")

In [ ]:
p = pstats.Stats('restats')
p.sort_stats("tottime").print_stats()

Profiling Jax based code doesnt work well with cProfile as it compiles the functions.